# Hinode/XRT — Implementation / 구현

**Paper**: Golub, L., DeLuca, E., Austin, G., Bookbinder, J., Caldwell, D., Cheimets, P., *et al.* (2007). "The X-Ray Telescope (XRT) for the *Hinode* Mission." *Solar Physics* **243**, 63–86. DOI: 10.1007/s11207-007-0182-1.

This notebook implements three focused topics from the XRT instrument paper:

1. **Wolter-I grazing-incidence reflectivity** $R(E, \theta)$ — physical origin of the high-energy effective-area cutoff (Fig. 15).
2. **9-filter temperature response $\mathcal{R}_f(T)$** — schematic reproduction of Fig. 7 from per-filter passbands and a coronal emissivity model.
3. **Filter-ratio (isothermal) thermometry** — the everyday XRT temperature diagnostic, demonstrated on a synthetic isothermal corona.

이 노트북은 XRT 논문에서 세 가지 핵심 주제 — Wolter-I grazing-incidence 반사도, 9-필터 온도 응답 함수, 필터 비 등온 온도 진단 — 를 직접 시뮬레이션한다.

Throughout the notebook, code follows a Google-style docstring convention (English).
All numerical models below are *toy/illustrative*: they reproduce the qualitative shapes of the figures in the paper using simplified analytic forms. They are NOT a substitute for the full XRT calibration tables (e.g., the SolarSoft `xrt_make_response.pro` chain).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
rng = np.random.default_rng(seed=20070712)  # paper publication date

## Part 1: Wolter-I Grazing-Incidence Reflectivity / Wolter-I Grazing-Incidence 반사도

X-ray optical constants for a material are usually written $n = 1 - \delta + i\beta$ with $\delta, \beta \sim 10^{-3}$–$10^{-6}$. At a grazing angle $\theta$, the Fresnel reflectance for an unpolarized beam reduces (small $\theta$) to

$$
R(\theta, E) \approx \left|\dfrac{\theta - \sqrt{\theta^2 - 2\delta(E) + 2i\beta(E)}}{\theta + \sqrt{\theta^2 - 2\delta(E) + 2i\beta(E)}}\right|^2.
$$

There is a critical angle $\theta_c \approx \sqrt{2\delta}$ above which $R$ collapses. For a smooth heavy-element surface (e.g., Zerodur ≈ SiO₂-rich glass), one approximation is

$$
\theta_c(E) \approx 0.93^\circ \,\dfrac{\sqrt{\rho\, Z/A}}{E[\text{keV}]} \quad (\rho \text{ in g/cm}^3).
$$

For a Wolter-I two-bounce design, the throughput is $R^2$.

Wolter-I 망원경은 두 번의 grazing 반사를 거치므로 throughput 은 $R^2$ 가 된다. 임계각 $\theta_c \propto 1/E$ 가 고에너지 차단을 결정한다.

In [ ]:
def grazing_reflectance(energy_kev, theta_deg, delta_at_1kev=1.0e-3, beta_at_1kev=1.0e-4):
    """Compute Fresnel reflectance at grazing incidence for a single bounce.

    Uses a simple energy scaling for the X-ray refractive-index decrement
    delta(E) = delta_1keV / E^2 and absorption beta(E) = beta_1keV / E^3,
    valid above ionisation edges. The form is intentionally illustrative and
    reproduces the qualitative shape of bare-Zerodur reflectance.

    Args:
        energy_kev: Photon energy in keV (scalar or numpy array).
        theta_deg: Grazing angle in degrees (scalar or numpy array).
        delta_at_1kev: Refractive-index decrement at 1 keV.
        beta_at_1kev: Absorption coefficient at 1 keV.

    Returns:
        np.ndarray: Real-valued reflectance R in [0, 1].
    """
    e = np.asarray(energy_kev, dtype=float)
    theta_rad = np.deg2rad(np.asarray(theta_deg, dtype=float))
    delta = delta_at_1kev / np.square(e)
    beta = beta_at_1kev / np.power(e, 3)
    z2 = np.square(theta_rad) - 2.0 * delta + 2j * beta
    z = np.sqrt(z2)
    r = (theta_rad - z) / (theta_rad + z)
    return np.abs(r) ** 2


def critical_angle_deg(energy_kev, delta_at_1kev=1.0e-3):
    """Approximate critical grazing angle theta_c = sqrt(2 delta).

    Args:
        energy_kev: Photon energy in keV.
        delta_at_1kev: Refractive-index decrement at 1 keV.

    Returns:
        float | np.ndarray: Critical angle in degrees.
    """
    delta = delta_at_1kev / np.square(np.asarray(energy_kev, dtype=float))
    return np.rad2deg(np.sqrt(2.0 * delta))

In [ ]:
# Plot single-bounce R(E) at three grazing angles representative of XRT shells
energies = np.linspace(0.2, 3.0, 400)
angles = [0.4, 0.6, 0.8]  # degrees

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

for theta in angles:
    R = grazing_reflectance(energies, theta)
    ax[0].plot(energies, R, label=f"theta = {theta:.1f} deg (single bounce)")
    ax[1].plot(energies, R ** 2, label=f"theta = {theta:.1f} deg (Wolter-I, two bounces)")

for a, title in zip(ax, ["Single-bounce R(E)", "Two-bounce R^2(E) — Wolter-I throughput"]):
    a.set_xlabel("Photon energy E [keV]")
    a.set_ylabel("Reflectance")
    a.set_xlim(0.2, 3.0)
    a.set_ylim(0, 1.05)
    a.set_title(title)
    a.grid(alpha=0.3)
    a.legend()

plt.tight_layout()
plt.show()

print("Critical angle at 0.5 / 1.0 / 1.5 keV [deg]:",
      np.round(critical_angle_deg(np.array([0.5, 1.0, 1.5])), 3))

Above ≈1.5 keV the two-bounce throughput drops steeply — exactly the behaviour seen in Fig. 15 of the paper. This sets the high-energy cutoff of any grazing-incidence design.

≈1.5 keV 위에서 두 번 반사 throughput 이 급감하는 것은 논문 Fig. 15 의 핵심 관측이며, 모든 grazing-incidence 설계가 본질적으로 high-energy 차단되는 이유이다.

## Part 2: 9-Filter Temperature Response — Schematic Fig. 7 / 9-필터 온도 응답

The XRT temperature response per filter is

$$
\mathcal{R}_f(T) = \int A_{\text{eff},f}(\lambda)\,\varepsilon(\lambda, T)\,d\lambda,
$$

where $A_{\text{eff},f}(\lambda) = A_\text{geom} \cdot R^2(\lambda) \cdot T_\text{pre}(\lambda) \cdot T_f(\lambda) \cdot \text{QE}(\lambda)$ and $\varepsilon(\lambda, T)$ is the per-EM emissivity (CHIANTI/APEC).

Below we (a) build a toy emissivity $\varepsilon(\lambda, T)$ from a thermal bremsstrahlung continuum plus a few cooling-loop lines, (b) construct simplified passbands $A_{\text{eff},f}(\lambda)$ that imitate XRT's nine filters by tuning their low-energy and high-energy cutoffs, and (c) plot $\mathcal{R}_f(T)$ against $\log T$ to reproduce qualitatively the curves of Fig. 7.

이 절은 thermal bremsstrahlung + 핵심 코로나 line 을 합성한 toy emissivity $\varepsilon(\lambda,T)$ 와 9 개 필터의 단순화된 passband 를 곱하여 Fig. 7 의 9 곡선을 정성적으로 재현한다.

In [ ]:
def toy_coronal_emissivity(wavelength_a, log_temperature_k):
    """Compute a toy coronal emissivity epsilon(lambda, T).

    The model contains an exponential bremsstrahlung continuum normalized to the
    plasma temperature, plus three coronal line groups (Fe IX/X, Fe XV/XVI,
    Fe XXIV) modelled as Gaussians in wavelength. Amplitudes are crudely
    weighted by Gaussian temperature kernels around their formation
    temperatures. The form is illustrative only — for any real analysis use
    CHIANTI / APEC.

    Args:
        wavelength_a: Wavelength grid in Angstrom (1-D array).
        log_temperature_k: log10(T [K]), scalar or 1-D array.

    Returns:
        np.ndarray: Emissivity, shape (n_T, n_lambda), in arbitrary units.
    """
    wave = np.atleast_1d(np.asarray(wavelength_a, dtype=float))
    log_t = np.atleast_1d(np.asarray(log_temperature_k, dtype=float))

    # k_B * T expressed as photon energy in keV: T[K] / 1.16e7 -> keV
    kt_kev = (10.0 ** log_t) / 1.16e7
    # photon energy (keV) corresponding to each wavelength (12.4 keV.A constant)
    e_kev = 12.398 / wave

    # bremsstrahlung continuum: dN/dE ~ exp(-E/kT)/sqrt(T)
    cont = (np.exp(-e_kev[None, :] / kt_kev[:, None]) /
            np.sqrt(10.0 ** log_t)[:, None])

    # three line complexes (peak wavelength, sigma in A, T_form, T_sigma)
    lines = [
        (171.0, 1.5, 6.0, 0.15),   # Fe IX/X 17.1 nm  (cool)
        (28.4, 0.6, 6.4, 0.20),    # Fe XV  ~28 A     (warm)
        (10.0, 0.4, 6.7, 0.30),    # Fe XVII ~9-10 A  (hot AR)
        (5.0, 0.3, 7.2, 0.35),     # Fe XXIV ~5 A     (flare)
    ]
    line_emiss = np.zeros_like(cont)
    for w0, sw, t0, st in lines:
        wave_kernel = np.exp(-0.5 * ((wave - w0) / sw) ** 2)
        temp_kernel = np.exp(-0.5 * ((log_t - t0) / st) ** 2)
        line_emiss += np.outer(temp_kernel, wave_kernel)

    return 0.3 * cont + 1.0 * line_emiss


def toy_filter_passband(wavelength_a, lambda_low, lambda_high, peak):
    """Compute a simplified XRT filter effective area A_eff,f(lambda).

    Models each XRT channel as a smooth top-hat between two cutoffs with a
    given peak effective area. Realistic XRT passbands have additional edge
    structure from filter K/L edges; this is sufficient for qualitative
    reproduction of Fig. 7's curve shapes.

    Args:
        wavelength_a: Wavelength grid in Angstrom.
        lambda_low: Short-wavelength cutoff [A].
        lambda_high: Long-wavelength cutoff [A].
        peak: Peak effective area [cm^2].

    Returns:
        np.ndarray: A_eff,f(lambda) on the input grid.
    """
    wave = np.asarray(wavelength_a, dtype=float)
    edge = 0.5 * (np.tanh((wave - lambda_low) / 0.6) -
                  np.tanh((wave - lambda_high) / 0.6))
    return peak * edge / np.max(edge)

In [ ]:
# XRT filters approximated by their nominal X-ray passbands (low, high in A) and peak A_eff (cm^2)
# Approximate ordering follows Table 4 / Fig. 17: A=Al-mesh ... I=Be-thick
filters = {
    "A: Al-mesh":  (4.0, 65.0, 1.10),
    "B: Al-poly":  (4.0, 50.0, 1.00),
    "C: C-poly":   (4.5, 45.0, 0.50),
    "D: Ti-poly":  (4.5, 35.0, 0.45),
    "E: Be-thin":  (5.5, 35.0, 0.40),
    "F: Be-med":   (6.0, 28.0, 0.30),
    "G: Al-med":   (4.0, 18.0, 0.10),
    "H: Al-thick": (3.5, 14.0, 0.05),
    "I: Be-thick": (3.5, 10.0, 0.005),
}

wave_grid = np.linspace(2.0, 200.0, 1500)
log_t_grid = np.linspace(5.5, 8.0, 60)
epsilon = toy_coronal_emissivity(wave_grid, log_t_grid)  # shape (n_T, n_lambda)

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: filter passbands
for name, (lo, hi, peak) in filters.items():
    aeff = toy_filter_passband(wave_grid, lo, hi, peak)
    ax[0].semilogy(wave_grid, np.maximum(aeff, 1e-4), label=name)
ax[0].set_xlabel("Wavelength [A]")
ax[0].set_ylabel("A_eff,f [cm^2]")
ax[0].set_title("Toy 9-filter A_eff,f(lambda)  (cf. Fig. 17)")
ax[0].set_xlim(2, 100)
ax[0].set_ylim(1e-3, 2.0)
ax[0].grid(alpha=0.3)
ax[0].legend(fontsize=8, ncol=2)

# Right: temperature response R_f(T)
responses = {}
for name, (lo, hi, peak) in filters.items():
    aeff = toy_filter_passband(wave_grid, lo, hi, peak)
    R_T = np.trapezoid(aeff[None, :] * epsilon, wave_grid, axis=1)
    responses[name] = R_T
    ax[1].semilogy(log_t_grid, np.maximum(R_T, 1e-12), label=name)

ax[1].set_xlabel("log T [K]")
ax[1].set_ylabel("R_f(T) [arbitrary]")
ax[1].set_title("Toy 9-filter temperature response (cf. Fig. 7)")
ax[1].set_xlim(5.5, 8.0)
ax[1].grid(alpha=0.3)
ax[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

The thinnest filters peak around log T ≈ 6.0–6.4 (Al-mesh, Al-poly, C-poly) and the thickest at log T ≈ 7.0–7.5 (Al-thick, Be-thick), reproducing the broad temperature span of Fig. 7. The XRT design covers log T ≈ 6.1–7.5 — the diagnostic window that motivates the nine-channel set.

가장 얇은 필터는 log T ≈ 6.0–6.4 에서 피크하고, 가장 두꺼운 필터는 log T ≈ 7.0–7.5 에서 피크하여 Fig. 7 의 넓은 T 범위를 정성적으로 재현한다.

## Part 3: Filter-Ratio (Isothermal) Thermometry / 필터 비 온도 진단

If the corona within an XRT pixel is isothermal at temperature $T$,

$$
\xi(T') = \text{EM}\,\delta(T' - T) \;\Longrightarrow\;
\dfrac{S_{f_1}}{S_{f_2}} = \dfrac{\mathcal{R}_{f_1}(T)}{\mathcal{R}_{f_2}(T)}.
$$

The EM cancels, leaving a function only of $T$. Where this ratio is monotonic, a measured ratio uniquely determines $T$.

다음 시뮬레이션에서는 (a) 두 필터 (Al-poly / Ti-poly) 의 비 곡선을 만들고, (b) 알려진 isothermal $\log T = 6.5$ corona 의 합성 신호에 Poisson noise 를 더한 뒤, (c) 비를 곡선에 대입하여 $T$ 를 회복한다.

In [ ]:
def filter_ratio_curve(t_response_a, t_response_b, log_t_grid, t_min=6.0, t_max=7.0):
    """Build the filter-ratio temperature diagnostic over a chosen T range.

    Args:
        t_response_a: R_a(T) sampled on log_t_grid.
        t_response_b: R_b(T) sampled on log_t_grid.
        log_t_grid: log10(T [K]) sample points.
        t_min: Lower log T bound for the monotonic diagnostic window.
        t_max: Upper log T bound.

    Returns:
        tuple: (log_t_window, ratio_window) with monotonic behaviour suitable
        for inversion.
    """
    mask = (log_t_grid >= t_min) & (log_t_grid <= t_max)
    return log_t_grid[mask], (t_response_a[mask] / t_response_b[mask])


def invert_ratio(measured_ratio, log_t_curve, ratio_curve):
    """Invert a measured filter ratio for the temperature.

    Uses linear interpolation on the (assumed monotonic) ratio curve. If the
    curve is not monotonic over the requested range, this will silently return
    one of multiple possible solutions — the caller should restrict the range
    in `filter_ratio_curve` to enforce uniqueness.

    Args:
        measured_ratio: Observed S_a/S_b (scalar or array).
        log_t_curve: log T values from `filter_ratio_curve`.
        ratio_curve: Ratio values from `filter_ratio_curve`.

    Returns:
        np.ndarray: Recovered log T.
    """
    # ensure monotonicity by sorting on the ratio axis
    order = np.argsort(ratio_curve)
    return np.interp(measured_ratio, ratio_curve[order], log_t_curve[order])

In [ ]:
# (a) build the ratio curve from Part 2's responses
key_a, key_b = "B: Al-poly", "D: Ti-poly"
log_t_window, ratio_window = filter_ratio_curve(
    responses[key_a], responses[key_b], log_t_grid, t_min=6.0, t_max=7.2
)

# (b) generate a synthetic isothermal observation at known T
true_log_t = 6.50
true_em = 1.0  # arbitrary unit
i_true = np.argmin(np.abs(log_t_grid - true_log_t))
expected_signal_a = responses[key_a][i_true] * true_em
expected_signal_b = responses[key_b][i_true] * true_em
# add Poisson-like noise (5% relative)
n_trials = 500
obs_a = rng.normal(expected_signal_a, 0.05 * expected_signal_a, size=n_trials)
obs_b = rng.normal(expected_signal_b, 0.05 * expected_signal_b, size=n_trials)
obs_ratio = obs_a / obs_b

# (c) recover T from each trial
recovered_log_t = invert_ratio(obs_ratio, log_t_window, ratio_window)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].plot(log_t_window, ratio_window, lw=2)
ax[0].axvline(true_log_t, color="k", ls="--", alpha=0.5, label=f"true log T = {true_log_t}")
ax[0].axhline(expected_signal_a / expected_signal_b, color="r", ls=":",
              label=f"expected ratio = {expected_signal_a/expected_signal_b:.2f}")
ax[0].set_xlabel("log T [K]")
ax[0].set_ylabel(f"R({key_a}) / R({key_b})")
ax[0].set_title("Filter-ratio diagnostic curve")
ax[0].grid(alpha=0.3)
ax[0].legend()

ax[1].hist(recovered_log_t, bins=30, color="steelblue", edgecolor="k", alpha=0.7)
ax[1].axvline(true_log_t, color="k", ls="--", label=f"true log T = {true_log_t}")
ax[1].axvline(np.mean(recovered_log_t), color="r", ls="-",
              label=f"recovered mean = {np.mean(recovered_log_t):.3f}")
ax[1].set_xlabel("recovered log T [K]")
ax[1].set_ylabel("trials")
ax[1].set_title(f"Monte-Carlo inversion ({n_trials} trials, 5% noise)")
ax[1].grid(alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"true     log T = {true_log_t:.4f}")
print(f"mean     log T = {np.mean(recovered_log_t):.4f}")
print(f"median   log T = {np.median(recovered_log_t):.4f}")
print(f"std       log T = {np.std(recovered_log_t):.4f}")

With 5%-level noise on the two filter signals, the recovered $\log T$ has standard deviation comparable to XRT's specification of $\Delta\log T = 0.2$ (Table 2 row 4). The bias depends on where on the ratio curve the operating point sits — flatter curves give larger errors.

5% 잡음에서 회복된 $\log T$ 의 표준편차는 XRT 사양 $\Delta\log T = 0.2$ (Table 2 line 4) 와 비교 가능한 수준이며, ratio 곡선의 기울기가 평탄한 영역에서는 오차가 커진다.

## Summary / 요약

| Concept / 개념 | This paper / 이 논문 | Modern equivalent / 현대 동등물 |
|---|---|---|
| Wolter-I two-bounce reflectance | Bare Zerodur, generalized asphere; Fig. 15 | NuSTAR multilayer optics; Solar Orbiter STIX (no GI) |
| 9-filter T response, Fig. 7 | Computed via APEC/ATOMDB (Smith+ 2001) | SolarSoft `xrt_make_response.pro`; AIA `aia_get_response.pro` |
| Filter-ratio T diagnostic | Routine use; e.g., Al-mesh / Ti-poly | Multi-band AIA ratios; CHIANTI-based DEM inversion |
| ≥6 channels for DEM | Established by Monte Carlo, Fig. 18 | AIA's six EUV channels enable similar DEM analysis |
| 0.92″ on-axis PRF | XRCF measurement, Fig. 10–11 | AIA 1.5″/pixel; Solar Orbiter EUI/HRI 0.5″/pixel |
| End-to-end calibration | XRCF (NASA) + XACT (Palermo) | All modern X-ray missions still use ground-vacuum facilities |

**Limitations of this notebook / 이 노트북의 한계.** All passbands and emissivities are toy analytic models. For quantitative analysis of XRT data, use the SolarSoft XRT package (`xrt_prep`, `xrt_make_response`, `xrt_dem_iterative2`) which carries the full ground-calibration tables, contamination model (Narukage et al. 2011, 2014), and CCD QE.

본 노트북의 모든 passband 와 emissivity 는 정성적 toy 모형이며, 실제 정량 분석에서는 SolarSoft XRT 패키지 (`xrt_prep`, `xrt_make_response`, `xrt_dem_iterative2`) 를 사용해야 한다.